In [ ]:
import asyncio
import json
import os
from typing import Annotated, Any, Never

from agent_framework import (
    AgentExecutor,
    AgentExecutorRequest,
    AgentExecutorResponse,
    Message,
    WorkflowBuilder,
    WorkflowContext,
    executor,
    tool,
)
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from dotenv import load_dotenv
from IPython.display import HTML, display
from pydantic import BaseModel

print("✅ All imports successful!")


## ကောက်နှုတ်ထားသော အထွေထွေ အဖြေများအတွက် Pydantic မော်ဒယ်များ သတ်မှတ်ရန် အဆင့် ၁

ဒီမော်ဒယ်တွေက အေးဂျင့်တွေပြန်ပေးမယ့် **schema** ကို သတ်မှတ်တယ်။ Pydantic ကို `response_format` နဲ့ အသုံးပြုခြင်းက အချက်ပေးတာတွေမှာ:
- ✅ အမျိုးအစား-လုံခြုံသော ဒေတာ ကောက်နုတ်မှု
- ✅ အလိုအလျောက် စစ်ဆေးခြင်း
- ✅ စာသားတမ်းဖြေကြားချက်တွေကနေ ဖတ်ရှုမှားယွင်းမှု မရှိစေခြင်း
- ✅ အကွာအခြား လမ်းကြောင်းများကို လွယ်ကူစွာ ခွဲခြားကြောင်း သတ်မှတ်စေခြင်း


In [ ]:
class BookingCheckResult(BaseModel):
    """Result from checking hotel availability at a destination."""

    destination: str
    has_availability: bool
    message: str


class AlternativeResult(BaseModel):
    """Suggested alternative destination when no rooms available."""

    alternative_destination: str
    reason: str


class BookingConfirmation(BaseModel):
    """Booking suggestion when rooms are available."""

    destination: str
    action: str
    message: str


print("✅ Pydantic models defined:")
print("   - BookingCheckResult (availability check)")
print("   - AlternativeResult (alternative suggestion)")
print("   - BookingConfirmation (booking confirmation)")

## ခြေလှမ်း ၂: ဟိုတယ်စာရင်း工具ကို ဖန်တီးပါ

ဒီ工具ကို **availability_agent** က အခန်းများ ရရှိနိုင်သလား စစ်ဆေးဖို့ ခေါ်ယူမှာ ဖြစ်ပါတယ်။ `@ai_function` decorator ကို အသုံးပြုပါတယ် -
- Python function ကို AI-ခေါ်ယူနိုင်တဲ့ tools အဖြစ် ပြောင်းလဲရန်
- LLM အတွက် JSON schema ကို အလိုအလျောက် ဖန်တီးရန်
- parameter စစ်ဆေးမှု ကို ကိုင်တွယ်ရန်
- လုပ်ကိုင်သူများက အလိုအလျှောက် ခေါ်ယူနိုင်ဖို့ ခွင့်ပြုရန်

ဒီ demo အတွက် -
- **Stockholm, Seattle, Tokyo, London, Amsterdam** → အခန်းများ ရှိပါတယ် ✅
- **အခြားမြို့များ အားလုံး** → အခန်း မရှိပါ ❌


In [ ]:
@tool(description="Check hotel room availability for a destination city")
def hotel_booking(destination: Annotated[str, "The destination city to check for hotel rooms"]) -> str:
    """
    Simulates checking hotel room availability.

    Returns JSON string with availability status.
    """
    display(
        HTML(f"""
        <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
            <strong>🔍 Tool Invoked:</strong> hotel_booking("{destination}")
        </div>
    """)
    )

    # Simulate availability check
    cities_with_rooms = ["stockholm", "seattle", "tokyo", "london", "amsterdam"]
    has_rooms = destination.lower() in cities_with_rooms

    result = {"has_availability": has_rooms, "destination": destination}

    return json.dumps(result)


print("✅ hotel_booking tool created with @tool decorator")

## အဆင့် ၃: လမ်းညွှန်ရေးအတွက် အခြေအနေ ဖွင့်ဆိုချက်ဖန်တီးခြင်း

ဤဖန်တီးထားသောလုပ်ဆောင်ချက်များသည် agent ၏ တုံ့ပြန်ချက်ကို စစ်ဆေးပြီး workflow အတွင်း တဆက်တည်း သွားရမည့်လမ်းကို ဆုံးဖြတ်ပေးသည်။

**အဓိက ပုံစံ:**
1. မက်ဆေ့ခ်ျသည် `AgentExecutorResponse` ဖြစ်မခြေစစ်ဆေးပါ
2. စနစ်တကျ ဖော်ပြထားသော ထွက်ရှိချက် (Pydantic မော်ဒယ်) ကို ပျားထုတ်ပါ
3. လမ်းညွှန်မှုအတွက် `True` သို့မဟုတ် `False` အဖြေပြန်ပါ

လုပ်ငန်းစဉ်သည် အဆက်အစပ်ပေါ်တွင် ဤအခြေအနေများကို မှတ်မိ၍ နောက်တစ်ဆင့်ခေါ်မည့် executor ကို ဆုံးဖြတ်မည်ဖြစ်သည်။


In [ ]:
def has_availability_condition(message: Any) -> bool:
    """
    Condition for routing when hotels ARE available.
    
    Returns True if the destination has hotel rooms.
    """
    if not isinstance(message, AgentExecutorResponse):
        return True  # Default to True if unexpected type

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        display(
            HTML(f"""
            <div style='padding: 12px; background: #c8e6c9; border-left: 4px solid #4caf50; border-radius: 4px; margin: 10px 0;'>
                <strong>✅ Condition Check:</strong> has_availability = <strong>{result.has_availability}</strong> for {result.destination}
            </div>
        """)
        )

        return result.has_availability
    except Exception as e:
        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                <strong>⚠️  Error:</strong> {str(e)}
            </div>
        """)
        )
        return False


def no_availability_condition(message: Any) -> bool:
    """
    Condition for routing when hotels are NOT available.
    
    Returns True if the destination has no hotel rooms.
    """
    if not isinstance(message, AgentExecutorResponse):
        return False

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffecb3; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
                <strong>❌ Condition Check:</strong> no_availability for {result.destination}
            </div>
        """)
        )

        return not result.has_availability
    except Exception as e:
        return False


print("✅ Condition functions defined:")
print("   - has_availability_condition (routes when rooms exist)")
print("   - no_availability_condition (routes when no rooms)")

## အဆင့် ၄: စိတ်ကြိုက်ပြသခြင်း အမိန့်ဆောင် ဖန်တီးခြင်း

အမိန့်ဆောင်များသည် ပြောင်းလဲမှုများ သို့မဟုတ် ဘက်လက်အကျိုးသက်ရောက်မှုများကို ဆောင်ရွက်သည့် workflow အစိတ်အပိုင်းများဖြစ်သည်။ နောက်ဆုံးရလဒ်ကို ပြသသော စိတ်ကြိုက်အမိန့်ဆောင်အား `@executor` decorator ကို အသုံးပြု၍ ဖန်တီးသည်။

**အဓိကအယူအဆများ-**
- `@executor(id="...")` - function ကို workflow အမိန့်ဆောင်အဖြစ် မှတ်ပုံတင်သည်
- `WorkflowContext[Never, str]` - input/output အတွက် အမျိုးအစား လမ်းညွှန်ချက်များ
- `ctx.yield_output(...)` - နောက်ဆုံး workflow ရလဒ်ကို yield ပြုသည်


In [ ]:
@executor(id="display_result")
async def display_result(response: AgentExecutorResponse, ctx: WorkflowContext[Never, str]) -> None:
    """
    Display the final result as workflow output.
    
    This executor receives the final agent response and yields it as the workflow output.
    """
    display(
        HTML("""
        <div style='padding: 15px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 4px; margin: 10px 0;'>
            <strong>📤 Display Executor:</strong> Yielding workflow output
        </div>
    """)
    )

    await ctx.yield_output(response.agent_run_response.text)


print("✅ display_result executor created with @executor decorator")

## အဆင့် ၅: ပတ်ဝန်းကျင်ဗားရီးများကို ထည့်သွင်းပါ

LLM client ကို ဖွဲ့စည်းပါ။ ဤနမူနာသည် အောက်ပါများနှင့် အလုပ်လုပ်သည်။
- **Microsoft Foundry** / **Azure OpenAI** (Responses API — အကြံပြုထားသည်)
- **Azure OpenAI**
- **OpenAI**


In [ ]:
# Load environment variables
load_dotenv()

# Configure the Microsoft Foundry provider with keyless authentication
provider = FoundryChatClient(
    project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
    credential=AzureCliCredential(),
)


## အဆင့် ၆: ဖွဲ့စည်းထားသော output များဖြင့် AI အေးဂျင့်များ ဖန်တီးခြင်း

ကျွန်ုပ်တို့သည် `AgentExecutor` ဖြင့် ထုပ်ပိုးထားသော **အထူးပြု အေးဂျင့်သုံးယောက်** ဖန်တီးသည်။

၁။ **availability_agent** - ကိရိယာကို အသုံးပြု၍ ဟိုတယ်အလဲနေမှုကို စစ်ဆေးသည်
၂။ **alternative_agent** - အခန်းမရှိပါက အခြားမြို့များကို အကြံပြုသည်
၃။ **booking_agent** - အခန်းရှိပါက အပ်မက်ရန် ထိန်းသိမ်းပေးသည်

**အဓိက လက္ခဏာများ။**
- `tools=[hotel_booking]` - အေးဂျင့်အား ကိရိယာပေးသည်
- `response_format=PydanticModel` - ဖွဲ့စည်းထားသော JSON output ကို အတိအကျ မောင်းနှင်သည်
- `AgentExecutor(..., id="...")` - workflow ၌ အသုံးပြုရန်အတွက် အေးဂျင့်ကို ထုပ်ပိုးသည်


In [ ]:
# Agent 1: Check availability with tool
availability_agent = AgentExecutor(
    provider.as_agent(
        name="availability-agent",
        instructions=(
            "You are a hotel booking assistant that checks room availability. "
            "Use the hotel_booking tool to check if rooms are available at the destination. "
            "Return JSON with fields: destination (string), has_availability (bool), and message (string). "
            "The message should summarize the availability status."
        ),
        tools=[hotel_booking],
        default_options={"response_format": BookingCheckResult},
    ),
    id="availability_agent",
)

# Agent 2: Suggest alternative (when no rooms)
alternative_agent = AgentExecutor(
    provider.as_agent(
        name="alternative-agent",
        instructions=(
            "You are a helpful travel assistant. When a user cannot find hotels in their requested city, "
            "suggest an alternative nearby city that has availability. "
            "Return JSON with fields: alternative_destination (string) and reason (string). "
            "Make your suggestion sound appealing and helpful."
        ),
        default_options={"response_format": AlternativeResult},
    ),
    id="alternative_agent",
)

# Agent 3: Suggest booking (when rooms available)
booking_agent = AgentExecutor(
    provider.as_agent(
        name="booking-agent",
        instructions=(
            "You are a booking assistant. The user has found available hotel rooms. "
            "Encourage them to book by highlighting the destination's appeal. "
            "Return JSON with fields: destination (string), action (string), and message (string). "
            "The action should be 'book_now' and message should be encouraging."
        ),
        default_options={"response_format": BookingConfirmation},
    ),
    id="booking_agent",
)

display(
    HTML("""
    <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
        <strong>✅ Created 3 Agents:</strong>
        <ul style='margin: 10px 0 0 0;'>
            <li><strong>availability_agent</strong> - Checks availability with hotel_booking tool</li>
            <li><strong>alternative_agent</strong> - Suggests alternative cities</li>
            <li><strong>booking_agent</strong> - Encourages booking</li>
        </ul>
    </div>
""")
)


## ဒုတိယအဆင့် ၇: အခြေအနေဆိုင်ရာ အားပြန်ချိတ်ဆက်မှုဖြင့် Workflow ကို တည်ဆောက်ခြင်း

ယခု `WorkflowBuilder` ကို အသုံးပြုပြီး အခြေအနေတစ်ရပ်ရပ်အလိုက် ချိတ်ဆက်ထားသော ဂရပ်ဖ်ကို တည်ဆောက်ပါမည်။

**Workflow ဖွဲ့စည်းပုံ:**
```
availability_agent (START)
        ↓
   Evaluate conditions
        ↙         ↘
[no_availability]  [has_availability]
        ↓              ↓
alternative_agent  booking_agent
        ↓              ↓
    display_result ←───┘
```

**အဓိက နည်းလမ်းများ:**
- `.set_start_executor(...)` - စတင်လုပ်ဆောင်သူကို သတ်မှတ်သည်
- `.add_edge(from, to, condition=...)` - အခြေအနေဆိုင်ရာ ချိတ်ဆက်မှုထည့်သွင်းသည်
- `.build()` - Workflow ကို အဆုံးသတ်စေသည်


In [ ]:
# Build the workflow with conditional routing
workflow = (
    WorkflowBuilder(
        start_executor=availability_agent,
        output_executors=[display_result],
    )
    # NO AVAILABILITY PATH
    .add_edge(availability_agent, alternative_agent, condition=no_availability_condition)
    .add_edge(alternative_agent, display_result)
    # HAS AVAILABILITY PATH
    .add_edge(availability_agent, booking_agent, condition=has_availability_condition)
    .add_edge(booking_agent, display_result)
    .build()
)

display(
    HTML("""
    <div style='padding: 20px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; border-radius: 8px; margin: 10px 0;'>
        <h3 style='margin: 0 0 15px 0;'>✅ Workflow Built Successfully!</h3>
        <p style='margin: 0; line-height: 1.6;'>
            <strong>Conditional Routing:</strong><br>
            • If <strong>NO availability</strong> → alternative_agent → display_result<br>
            • If <strong>availability</strong> → booking_agent → display_result
        </p>
    </div>
""")
)

## အဆင့် ၈: စမ်းသပ်မှုအမှု ၁ ကို လုပ်ဆောင်ပါ - ရရှိနိုင်မှု မရှိသော မြို့ (ပါရီ)

ပါရီမြို့ရှိ ဟိုတယ်များကို မေးမြန်းခြင်းဖြင့် **ရရှိနိုင်မှု မရှိခြင်း** လမ်းကြောင်းကို စမ်းသပ်ကြည့်ရအောင် (ကျွန်ုပ်တို့၏ အတုစမ်းသပ်မှုတွင် အခန်းများ မရှိပါ။)


In [ ]:
display(
    HTML("""
    <div style='padding: 20px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #e65100;'>🧪 TEST CASE 1: Paris (No Availability)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → alternative_agent → display_result</p>
    </div>
""")
)

# Create request for Paris
request_paris = AgentExecutorRequest(
    messages=[Message(role="user", text="I want to book a hotel in Paris")], should_respond=True
)

# Run the workflow
events_paris = await workflow.run(request_paris)
outputs_paris = events_paris.get_outputs()

# Display results
if outputs_paris:
    result_paris = AlternativeResult.model_validate_json(outputs_paris[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); border-radius: 12px; box-shadow: 0 4px 12px rgba(255,165,0,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0; color: #333;'>🏆 WORKFLOW RESULT (Paris)</h3>
            <div style='background: white; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ❌ No rooms in Paris</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Alternative Suggestion:</strong> 🏨 {result_paris.alternative_destination}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Reason:</strong> {result_paris.reason}</p>
            </div>
        </div>
    """)
    )

## အဆင့် ၉: စမ်းသပ်မှု ကိစ္စ ၂ ကို စစ်ဆေးပါ - မြို့နှင့် ရနိုင်မှု (စတော့(စ်)ဟုမ်း)

ယခု သင်္ကေတအရ စတောက်ဟိုမ္ (Stockholm) တွင် ဟိုတယ်များကို ရယူ၍ **ရနိုင်မှု** လမ်းကြောင်းကို စမ်းသပ်ကြမယ် (ယင်းမှာ စမ်းသပ်နေအိမ်ခန်းများ ရှိသည်)။


In [ ]:
display(
    HTML("""
    <div style='padding: 20px; background: #e8f5e9; border-left: 4px solid #4caf50; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #1b5e20;'>🧪 TEST CASE 2: Stockholm (Has Availability)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → booking_agent → display_result</p>
    </div>
""")
)

# Create request for Stockholm
request_stockholm = AgentExecutorRequest(
    messages=[Message(role="user", text="I want to book a hotel in Stockholm")], should_respond=True
)

# Run the workflow
events_stockholm = await workflow.run(request_stockholm)
outputs_stockholm = events_stockholm.get_outputs()

# Display results
if outputs_stockholm:
    result_stockholm = BookingConfirmation.model_validate_json(outputs_stockholm[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #4caf50 0%, #8bc34a 100%); color: white; border-radius: 12px; box-shadow: 0 4px 12px rgba(76,175,80,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0;'>🏆 WORKFLOW RESULT (Stockholm)</h3>
            <div style='background: white; color: #333; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ✅ Rooms Available!</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Destination:</strong> 🏨 {result_stockholm.destination}</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Action:</strong> {result_stockholm.action}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Message:</strong> {result_stockholm.message}</p>
            </div>
        </div>
    """)
    )

## အဓိကယူဆချက်များနှင့် နောက်တစ်ဆင့်များ

### ✅ သင်ယူခဲ့သောအရာများ

1. **WorkflowBuilder ပုံစံ**
   - ဝင်ပေါက်ကို သတ်မှတ်ရန် `.set_start_executor()` ကို အသုံးပြုပါ
   - အခြေအနေ စစ်ဆေးရန် `.add_edge(from, to, condition=...)` ကို အသုံးပြုပါ
   - workflow ကို အတည်ပြုရန် `.build()` ကို ခေါ်ပါ

2. **အခြေအနေစစ်ဆေးပြီး လမ်းညွှန်ခြင်း**
   - အခြေအနေလုပ်ဆောင်ချက်များသည် `AgentExecutorResponse` ကို စစ်တမ်းချနိင်သည်
   - လမ်းညွှန်ရေး ဆုံးဖြတ်ချက်များအတွက် ဖွဲ့စည်းထားသော အထွက်များကို ဖော်ထုတ်ပါ
   - လမ်းစဉ်တစ်ခုကို လှုပ်ရှားစေမည်ဆိုလျှင် `True` ကို ပြန်ပေးပါ၊ မလုပ်ဆောင်မည်ဆိုလျှင် `False` ကို ပြန်ပေးပါ

3. **ကိရိယာ ပေါင်းစည်းခြင်း**
   - Python function များကို AI ကိရိယာများအဖြစ် ပြောင်းလဲရန် `@ai_function` ကို အသုံးပြုပါ
   - Agents များသည်လိုအပ်သည့်အခါ ကိရိယာများကို အလိုအလျောက် ခေါ်ဆိုပါမည်
   - ကိရိယာများသည် agents များအတွက် JSON ကို ပြန်ပေးပါမည်

4. **ဖွဲ့စည်းထားသော အထွက်များ**
   - အမျိုးအစားလုံခြုံသော ဒေတာ ထုတ်ယူမှုအတွက် Pydantic မော်ဒယ်များကို အသုံးပြုပါ
   - agent များ ဖန်တီးစဉ်တွင် `response_format=MyModel` ကို သတ်မှတ်ပါ
   - `Model.model_validate_json()` ဖြင့် ပြန်လည်စစ်ဆေးပါ

5. **စိတ်ကြိုက် Executors များ**
   - workflow အစိတ်အပိုင်းများ ဖန်တီးရန် `@executor(id="...")` ကို အသုံးပြုပါ
   - Executors များသည် ဒေတာကို ပြောင်းလဲနိုင်သည် သို့မဟုတ် နောက်ဆက်တွဲ လုပ်ဆောင်ချက်များ ဖော်ဆောင်နိုင်သည်
   - workflow ရလဒ်များ ထုတ်ဖေါ်ရန် `ctx.yield_output()` ကို အသုံးပြုပါ

### 🚀 အမှန်တကယ် အသုံးချနိုင်သော နယ်ပယ်များ

- **ခရီးသွား စီစဉ်ခြင်း**: ရနိုင်မှု စစ်ဆေးခြင်း၊ အခြားရွေးချယ်စရာ ပြန်ညွှန်ခြင်း၊ ရွေးချယ်စရာများ နှိုင်းယှဉ်ခြင်း
- **ဖောက်သည်ဆက်ဆံရေး**: ပြဿနာအမျိုးအစား၊ စိတ်ခံစားချက်၊ ဦးစားပေးမှုအရ လမ်းညွှန်ခြင်း
- **အီလက်ထရွနစ် ကုန်သည်လုပ်ငန်း**: မလှောင်ပစ္စည်း စစ်ဆေးခြင်း၊ အခြားရွေးချယ်စရာ ပြန်ညွှန်ခြင်း၊ မှာယူခြင်းလုပ်ငန်းဆောင်တာ
- **အကြောင်းအရာ ထိန်းချုပ်ခြင်း**: အဆိပ်အတောက် အဆင့်များ၊ အသုံးပြုသူ အမှတ်အသားများအရ လမ်းညွှန်ခြင်း
- **အတည်ပြုမှု Workflow များ**: ပမာဏ၊ အသုံးပြုသူ အခန်းကဏ္ဍ၊ စွမ်းအင်အဆင့်အရ လမ်းညွှန်ခြင်း
- **အဆင့်အတန်း ဘာသာပြန် ချုပ်ဆိုခြင်း**: ဒေတာအရည်အသွေး၊ ပြည့်စုံမှုအရ လမ်းညွှန်ခြင်း

### 📚 နောက်တဆင့်များ

- ပိုမိုရှုပ်ထွေးသော အခြေအနေများ ထည့်သွင်းခြင်း (သီးခြား စည်းကမ်းများ အစု)
- workflow အခြေအနေ စီမံခန့်ခွဲမှုဖြင့် လည်ပတ်ရေး စနစ်ဖန်တီးခြင်း
- ထပ်ရလဒ်လုပ်စဉ်များအတွက် သက်ဆိုင်ရာ sub-workflows ထပ်ဖြည့်ခြင်း
- အမှန်တကယ် API များနှင့် ပေါင်းစည်းခြင်း (ဟိုတယ်မှာယူခြင်း၊ မလှောင်ပစ္စည်း စနစ်များ)
- အမှား ကိုင်တွယ်မှုနှင့် အစားထိုး လမ်းကြောင်းများ ထည့်သွင်းခြင်း
- ပြသနိုင်စွမ်း встроенный visualization ကိရိယာများဖြင့် workflow များကို ကြည့်ရှုခြင်း


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ပြောကြားချက်**
ဤစာတမ်းကို AI ဘာသာပြန်ဝန်ဆောင်မှု [Co-op Translator](https://github.com/Azure/co-op-translator) အသုံးပြု၍ ဘာသာပြန်ထားပါသည်။ ကျွန်ုပ်တို့သည် တိကျမှန်ကန်မှုအတွက် ကြိုးပမ်းနေသော်လည်း၊ စက်ကိရိယာဘာသာပြန်ခြင်းများတွင် အမှားများ သို့မဟုတ် မှားယွင်းချက်များ ပါဝင်နိုင်ကြောင်း သတိပြုပါရန် လိုအပ်ပါသည်။ မူလစာတမ်းကို မူရင်းဘာသာဖြင့်သာ ယုံကြည်စိတ်ချရသော အချက်အလက်အဖြစ် သတ်မှတ်သင့်သည်။ အရေးကြီးသည့် သတင်းအချက်အလက်များအတွက် ပရော်ဖက်ရှင်နယ် လူသားဘာသာပြန်သူဝန်ဆောင်မှုကို အကြံပြုပါသည်။ ဤဘာသာပြန်ချက်ကို အသုံးပြုခြင်းမှ ဖြစ်ပေါ်လာသော နားလည်မှုကွာခြားမှုများ သို့မဟုတ် မမှန်ကန်သော အသုံးပြုမှုများအတွက် ကျွန်ုပ်တို့ တာဝန်မခံပါ။
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
